<a href="https://colab.research.google.com/github/sizormohanty/Data-Science/blob/master/QuickTrip_Stores_Zip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
#!/usr/bin/env python3
"""
QuikTrip Store Zip Code Scraper
Fetches all QuikTrip store locations and extracts zip codes.
Outputs results to quicktrip_zipcodes.csv
"""

import requests
import csv
import json
import time
import re
from collections import Counter

# ── Session setup ──────────────────────────────────────────────────────────────
session = requests.Session()
session.headers.update({
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://www.quiktrip.com/",
})

# ── QuikTrip store locator API ─────────────────────────────────────────────────
# QT uses a Woosmap / proprietary store API; this mirrors the XHR the browser makes.
STORES_API = "https://storerocket.io/api/user/56wp4Nzx8b/stores"

def fetch_all_stores():
    """Fetch all QuikTrip store records from the store locator API."""
    stores = []
    params = {
        "radius": "5000",   # large radius to cast wide net
        "limit": "500",
        "units": "miles",
    }

    print("Fetching stores from QuikTrip store locator API …")
    try:
        resp = session.get(STORES_API, params=params, timeout=20)
        resp.raise_for_status()
        print(f"  API Response Status Code: {resp.status_code}")
        print(f"  API Response Headers: {resp.headers}")
        data = resp.json()
        if "results" in data and "stores" in data["results"]:
            stores = data["results"]["stores"]
            print(f"  → {len(stores)} stores returned from API")
    except json.JSONDecodeError as e:
        print(f"  StoreRocket API returned non-JSON response. Error: {e}")
        print(f"  Response content: {resp.text[:500]}...") # Print first 500 chars for debugging
    except Exception as e:
        print(f"  StoreRocket API failed: {e}")

    return stores


def extract_zip_from_address(address: str) -> str:
    """Pull a 5-digit (or ZIP+4) zip code from a free-text address string."""
    m = re.search(r"\b(\d{5})(?:-\d{4})?\b", address or "")
    return m.group(1) if m else ""


def parse_store(store: dict) -> dict:
    """Normalise a raw store record into a flat dict."""
    address_parts = [
        store.get("address", ""),
        store.get("address2", ""),
        store.get("city", ""),
        store.get("state", ""),
        store.get("postcode", store.get("zip", store.get("postal_code", ""))),
    ]
    full_address = ", ".join(p for p in address_parts if p)

    zipcode = (
        store.get("postcode")
        or store.get("zip")
        or store.get("postal_code")
        or extract_zip_from_address(full_address)
        or ""
    )
    # Keep only the 5-digit base
    zipcode = re.sub(r"-\d{4}$", "", str(zipcode).strip())

    return {
        "store_id":   store.get("id", ""),
        "name":       store.get("name", "QuikTrip"),
        "address":    store.get("address", ""),
        "city":       store.get("city", ""),
        "state":      store.get("state", ""),
        "zip":        zipcode,
        "phone":      store.get("phone", ""),
        "latitude":   store.get("lat", ""),
        "longitude":  store.get("lng", ""),
    }


def scrape_via_state_search():
    """
    Fallback: hit QT's own store-locator page for each US state abbreviation
    and scrape JSON embedded in the HTML or returned by the search endpoint.
    """
    STATES = [
        "AL","AZ","AR","CA","CO","CT","DE","FL","GA","ID","IL","IN","IA",
        "KS","KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV",
        "NH","NJ","NM","NY","NC","ND","OH","OK","OR","PA","RI","SC","SD",
        "TN","TX","UT","VT","VA","WA","WV","WI","WY",
    ]
    # QT operates mainly in: AZ, GA, IA, IL, IN, KS, MO, NC, NE, OK, SC, TN, TX, VA
    QT_STATES = ["AZ","GA","IA","IL","IN","KS","MO","NC","NE","OK","SC","TN","TX","VA"]

    stores = []
    SEARCH_URL = "https://www.quiktrip.com/api/stores/search"

    for state in QT_STATES:
        try:
            resp = session.get(
                SEARCH_URL,
                params={"state": state, "limit": 500},
                timeout=15,
            )
            print(f"  {state} API Response Status Code: {resp.status_code}")
            print(f"  {state} API Response Headers: {resp.headers}")
            if resp.status_code == 200:
                data = resp.json()
                batch = data if isinstance(data, list) else data.get("stores", [])
                print(f"  {state}: {len(batch)} stores")
                stores.extend(batch)
            time.sleep(0.4)   # be polite
        except json.JSONDecodeError as e:
            print(f"  {state}: API returned non-JSON response. Error: {e}")
            print(f"  {state}: Response content: {resp.text[:500]}...") # Print first 500 chars for debugging
        except Exception as e:
            print(f"  {state}: error – {e}")

    return stores


def main():
    # Use scrape_via_state_search as the primary data source due to primary API issues
    all_raw = scrape_via_state_search()

    if not all_raw:
        print("\n⚠  No store data retrieved from state-by-state search.")
        print("   QuikTrip may have changed their API. Try inspecting network")
        print("   requests on https://www.quiktrip.com/stores while searching")
        print("   for a location and update relevant endpoints above.\n")
        return

    # Deduplicate by store_id
    seen = set()
    records = []
    for raw in all_raw:
        parsed = parse_store(raw)
        key = parsed["store_id"] or f"{parsed['latitude']},{parsed['longitude']}"
        if key not in seen:
            seen.add(key)
            records.append(parsed)

    records.sort(key=lambda r: (r["state"], r["city"], r["zip"]))

    # ── Write full CSV ─────────────────────────────────────────────────────────
    out_full = "/mnt/user-data/outputs/quicktrip_stores.csv"
    with open(out_full, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(records[0].keys()))
        writer.writeheader()
        writer.writerows(records)

    # ── Write zip-codes-only CSV ───────────────────────────────────────────────
    zipcodes = sorted({r["zip"] for r in records if r["zip"]})
    out_zips = "/mnt/user-data/outputs/quicktrip_zipcodes.csv"
    with open(out_zips, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["zip_code"])
        for z in zipcodes:
            writer.writerow([z])

    # ── Summary ────────────────────────────────────────────────────────────────
    state_counts = Counter(r["state"] for r in records)
    print(f"\n{'='*50}")
    print(f"Total stores found : {len(records)}")
    print(f"Unique zip codes   : {len(zipcodes)}")
    print(f"\nStores by state:")
    for state, count in sorted(state_counts.items()):
        print(f"  {state}: {count}")
    print(f"\nFiles saved:")
    print(f"  Full store data → {out_full}")
    print(f"  Zip codes only  → {out_zips}")


if __name__ == "__main__":
    main()

  AZ API Response Status Code: 404
  AZ API Response Headers: {'Server': 'nginx', 'Content-Type': 'text/html; charset=UTF-8', 'Vary': 'Accept-Encoding', 'Expires': 'Wed, 11 Jan 1984 05:00:00 GMT', 'Cache-Control': 'no-cache, must-revalidate, max-age=0, no-store, private', 'Link': '<https://www.quiktrip.com/wp-json/>; rel="https://api.w.org/"', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains; preload', 'Content-Encoding': 'gzip', 'Content-Length': '15830', 'Date': 'Thu, 23 Apr 2026 23:14:26 GMT', 'Connection': 'keep-alive'}
  GA API Response Status Code: 404
  GA API Response Headers: {'Server': 'nginx', 'Content-Type': 'text/html; charset=UTF-8', 'Vary': 'Accept-Encoding', 'Expires': 'Wed, 11 Jan 1984 05:00:00 GMT', 'Cache-Control': 'no-cache, must-revalidate, max-age=0, no-store, private', 'Link': '<https://www.quiktrip.com/wp-json/>; rel="https://api.w.org/"', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains; preload', 'Content-Encoding': 'gzip', 'C